In [ ]:
# Upstream SimplePIR's own call sequence (real Setup, Query, Answer, Recover) on a seeded one-bit database,
# whose entries the library packs several to a Z_p element, against 8-bit entries, which do not pack.
# Why the S0-go adapter addresses packed entries itself for the README anchor (decision 21).
import os, subprocess
from pathlib import Path

repo = Path.cwd().parent
env = {**os.environ, "CC": subprocess.run(["brew", "--prefix", "llvm"], capture_output=True, text=True).stdout.strip() + "/bin/clang",
       "GOTOOLCHAIN": "local", "CGO_ENABLED": "1", "GOPROXY": "off"}
r = subprocess.run(["go", "run", "./cmd/packedcheck"], cwd=repo / "src/adapters/simplepir_go", env=env,
                   capture_output=True, text=True)
print("\n".join(l for l in r.stdout.splitlines() if l.startswith("entry_bits")))
assert r.returncode == 0, r.stderr[-2000:]

In [ ]:
# The anchors tier: S0-go at 1 GiB as 2^33 one-bit entries (SimplePIR README), S3 and S4 at 1 KiB
# (Colombo et al. Fig 6), at the spec's repetitions, on AC power with the lid open.
import json, sys, time
from collections import Counter

sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import hygiene, validate

assert hygiene.power_source() == "AC Power", f"plug the Mac in first ({hygiene.power_source()})"
assert hygiene.lid_closed() is False, "open the lid first: a closed lid sleeps the Mac despite caffeinate"
before = {p.name for p in (repo / "results/raw").glob("*_anchors_*")}
t0 = time.monotonic()
r = subprocess.run(["make", "anchors"], cwd=repo, capture_output=True, text=True)
run_dir = next(repo / "results/raw" / n for n in {p.name for p in (repo / "results/raw").glob("*_anchors_*")} - before)
meta = json.loads((run_dir / "meta.json").read_text())
print(f"make anchors: exit {r.returncode}, {(time.monotonic() - t0) / 60:.1f} min wall, run {run_dir.name}")
print({k: meta[k] for k in ("git_sha_harness", "platform", "tier")})
assert r.returncode == 0, r.stderr[-3000:]
assert meta["git_sha_harness"] and not meta["git_sha_harness"].endswith("-dirty"), "commit src/ and configs/ first"
bad = []
for f in sorted(run_dir.glob("*_*_*_*.jsonl")):
    rows = [json.loads(l) for l in f.read_text().splitlines()]
    probs = [str(p) for x in rows for p in validate.check_row(x, reference_platform=meta["platform"]) if p.severity == "error"]
    online = [x for x in rows if x["phase"] == "online" and x["status"] == "ok"]
    statuses = sorted({x["status"] for x in rows})
    wrong = sum(not x["correct"] for x in online)
    print(f"{f.stem:26s} {len(rows):3d} rows  {statuses}  wrong {wrong}  schema errors {len(probs)}"
          f"  peak {max((x['metrics'].get('process_peak_rss_bytes') or 0) for x in rows) / 2**30:.2f} GiB")
    if probs or wrong or statuses != ["ok"]:
        bad.append(f.stem)
assert not bad, bad

In [ ]:
# The targets these cells answer (T3 rows; the rest need the Tier 1 grid). R23 and R24 are exact
# (communication), R25 and R9 report the hardware factor over the paper's machine.
from cs4406_sem_project.analysis import frame, reproduce

df, status = frame.load_runs([run_dir], meta["platform"])
ids = ["R23", "R24", "R25", "R9-S4-1KiB", "R9-S3-1KiB", "R10-S4-1KiB", "R10-S3-1KiB", "R11-S4-1KiB", "R11-S3-1KiB"]
targets = [t for t in reproduce.load_targets() if t.id in ids]
t3 = reproduce.compare(df, targets).set_index("id").loc[ids]
print(t3[["cell", "ours", "ci_lo", "ci_hi", "unit", "target", "deviation", "verdict"]].round(3).to_string())